# 00 — Project Environment Setup

**Objective:** Mount Google Drive, set up directory structure, verify GPU availability, and save an environment snapshot.

Before running: Ensure the notebook is saved inside the `notebooks` directory in Google Drive. All persistent outputs are saved to Drive; only temporary files are kept under `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/DiffusionPolicyEvents')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Project root does not exist: {PROJECT_ROOT}'
    )
sys.path.insert(0, str(PROJECT_ROOT))

from src.project_paths import ProjectPaths
paths = ProjectPaths(PROJECT_ROOT).ensure()
print(paths)


## Working Principle

The code runs from `/content` when high performance is required, but all important artifacts are saved under `MyDrive/DiffusionPolicyEvents`. Do not delete Drive directories from within the notebook.

In [ ]:
import os, platform, subprocess, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(subprocess.check_output(['nvidia-smi'], text=True)[:2000])

In [ ]:
# Save package list for future restoration
import subprocess
snapshot = paths.root / 'environment'
snapshot.mkdir(exist_ok=True)
(snapshot / 'pip_freeze.txt').write_text(
    subprocess.check_output(['pip', 'freeze'], text=True), encoding='utf-8')
print('Environment snapshot saved.')

## Results Summary

At the end of the run, summarize:

- What was actually executed?
- Which environment was verified?
- Which important project paths were initialized?
- Which files were created and at what path?
- What hardware/software was detected?
- What failed or remains open?
- What is the next step?

In [ ]:
from src.run_logging import write_notebook_summary

summary = {
    'notebook': '00_Project_Setup.ipynb',
    'status': 'completed',
    'configuration': {
        'project_root': str(paths.root),
        'persistent_storage': 'Google Drive',
        'temporary_storage': '/content',
    },
    'seed': None,
    'outputs': [
        str(paths.root / 'environment' / 'pip_freeze.txt'),
    ],
    'metrics': {
        'python': platform.python_version(),
        'pytorch': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    'observations': (
        'Google Drive was mounted successfully. '
        'Project paths were initialized. '
        'GPU/CUDA availability was verified. '
        'Python package snapshot was saved.'
    ),
    'decision': (
        'Project environment setup validated. '
        'The persistent project root is Google Drive; /content is for temporary/high-performance work.'
    ),
    'next_step': (
        'Proceed to 01_Dataset_Audit_and_Splits.ipynb for raw dataset validation and deterministic split generation.'
    ),
}

summary_path = write_notebook_summary(paths.summaries, summary)
print(f'Summary saved to: {summary_path}')
summary